In [40]:
import numpy as np



#########返回的数组应为节点序号，不是节点索引#########


def H_Degree_SG(G_matrix,NodeList,SeedNodeNumber):
    #每个节点的超度
    HyperEdgeList = []

    for i in range(len(G_matrix)):
        HyperEdgeNumber = len(np.where(G_matrix[i] == 1)[0])
        HyperEdgeList.append(HyperEdgeNumber)

    #将HyperEdgeList从大到小排序后的索引
    HyperEdgeList_sort_index = np.argsort(-np.array(HyperEdgeList))
    SeedNodes = []

    for i in range(SeedNodeNumber):
        SeedNodes.append(NodeList[HyperEdgeList_sort_index[i]])

    return SeedNodes




def Degree_SG(G_matrix,NodeList,SeedNodeNumber):
    # 每个节点的度
    DegreeList = []

    for i in range(len(G_matrix)):
        HyperEdge_index = np.where(G_matrix[i] == 1)[0]
        #节点的邻居索引
        Neighbors_index = []

        for j in HyperEdge_index:
            Neighbors_index += list(np.where(G_matrix[:,j] == 1)[0])
        Neighbors_index = np.unique(Neighbors_index)
        DegreeList.append(len(Neighbors_index) - 1)

    # 将HyperEdgeList从大到小排序后的索引
    HyperEdgeList_sort_index = np.argsort(-np.array(DegreeList))
    SeedNodes = []

    for i in range(SeedNodeNumber):
        SeedNodes.append(NodeList[HyperEdgeList_sort_index[i]])

    return SeedNodes

In [41]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random


class QNetwork(nn.Module):
    """DQN网络结构"""
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()  # 调用父类 nn.Module 的构造函数，完成父类的初始化
        # 定义一个顺序容器，按顺序包含一系列的神经网络层
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.LayerNorm(256),  # 批量归一化
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
            # nn.LayerNorm(action_dim)
        )

    def forward(self, x):
        # 定义前向传播过程，将输入 x 依次通过 self.fc 中的所有层，最终返回网络的输出
        return self.fc(x)


class Compete_DQN:
    def __init__(self, epsilon, alpha, gamma, state_dim, action_dim, node_hyperedge_avgsize, avg_HigerOrder_dis_seednodes, candidate_nodes_pos=None, device='cpu'):
        self.q_net = QNetwork(state_dim, action_dim).to(device)  # 创建一个 QNetwork 实例，并将其移动到指定设备（如 GPU）上
        self.target_net = QNetwork(state_dim, action_dim).to(device)  # 创建一个目标网络，用于稳定训练过程，同样移动到指定设备上
        self.target_net.load_state_dict(self.q_net.state_dict())  # 将目标网络的参数初始化为与 Q 网络相同
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=alpha)  # 定义优化器，使用 Adam 优化算法来更新 Q 网络的参数，学习率为 alpha
        self.gamma = gamma  # 折扣因子，用于计算未来奖励的折扣值
        self.epsilon = epsilon  # 探索率，用于 ε-贪心策略，控制探索和利用的平衡
        self.state_dim = state_dim  # 当前状态的维度，由编码器编码当前状态决定
        self.action_dim = len(candidate_nodes_pos)  # 动作的维度，即可能的动作数量
        self.device = device  # 指定使用的设备，如 'cpu' 或 'cuda'
        self.loss_fn = nn.MSELoss()  # 定义损失函数，使用均方误差损失（MSE Loss）来计算预测值和目标值之间的误差
        self.node_hyperedge_avgsize = node_hyperedge_avgsize  # 获取nodes的平均超边大小
        self.avg_HigerOrder_dis_seednodes = avg_HigerOrder_dis_seednodes  # 获取nodes与seednodes的高阶距离列表
        self.candidate_nodes_pos = candidate_nodes_pos  # 候选节点的索引列表
        self.candidate_pos_to_index = {node: idx for idx, node in enumerate(candidate_nodes_pos)} # 候选节点索引到动作索引的映射字典

    def _encode_state(self, state):
        """基于超图的节点特征编码"""
        if state == -1:  # 初始状态
            return np.zeros(self.state_dim)
        else:
            features = []
            for node in state:
                hyperedge_avgsize = self.node_hyperedge_avgsize[node]  # 节点平均超边大小
                higerorder = self.avg_HigerOrder_dis_seednodes[node]  # 获取节点与种子节点的高阶距离
                features.extend([hyperedge_avgsize, higerorder])  # 节点平均超边大小
            features = features[:self.state_dim] + [0] * (self.state_dim - len(features))  # 截断或填充至固定长度
            return np.array(features)

    def take_action(self, state, CandidateNodesPool):
        state_feature = self._encode_state(state)  # 对当前状态进行编码，得到状态特征
        if np.random.rand() < self.epsilon:
            selected_node = np.random.choice(CandidateNodesPool)
            selected_index = self.candidate_pos_to_index[selected_node]  # 获取该节点在候选节点池中的索引
            return selected_node, selected_index  # 返回节点位置和索引
        else:
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state_feature).unsqueeze(0).to(self.device)  # 将状态特征转换为FloatTensor类型的张量，并增加一个维度
                q_values = self.q_net(state_tensor)
                candidate_indices = [self.candidate_pos_to_index[node] for node in CandidateNodesPool]  # 获取候选节点在动作空间中的索引
                valid_q_values = q_values[0, candidate_indices]  # 从所有动作的 Q 值中选取候选节点对应的 Q 值
                action_idx = torch.argmax(valid_q_values).item()  # 选择 Q 值最大的动作的索引
                selected_node = CandidateNodesPool[action_idx]  # 根据索引选择候选节点
                selected_index = self.candidate_pos_to_index[selected_node]  # 获取该节点在候选节点池中的索引
                return selected_node, selected_index  # 返回节点位置和索引

    def update(self, batch):
        states, actions, rewards, next_states, dones = batch  # 从采样的批次数据中解包出状态、动作、奖励、下一个状态和是否结束的标志
        states = torch.FloatTensor(np.array(states)).to(self.device)  # 将状态数据转换为NumPy数组，再转换为FloatTensor类型的张量，并移动到指定设备（CPU或GPU）上
        actions = torch.LongTensor(actions).unsqueeze(1).to(self.device)  # 将动作数据转换为LongTensor类型的张量，并增加一个维度，使其形状为 [batch_size, 1]，然后移动到指定设备上
        rewards = torch.FloatTensor(rewards).to(self.device)  # 将奖励数据转换为FloatTensor类型的张量，并移动到指定设备上
        next_states = torch.FloatTensor(np.array(next_states)).to(self.device)   # 将下一个状态数据转换为NumPy数组，再转换为FloatTensor类型的张量，并移动到指定设备上
        dones = torch.FloatTensor(dones).to(self.device)  # 将是否结束的标志数据转换为FloatTensor类型的张量，并移动到指定设备上

        # 计算当前Q值
        current_q = self.q_net(states).gather(1, actions)  # self.q_net(states) 会输出当前状态下所有动作的Q值, .gather(1, actions) 会根据 actions 中的索引，从所有动作的Q值中选取对应动作的Q值

        # 计算目标Q值
        # 关闭梯度计算，因为目标网络的参数在这一步不需要更新
        with torch.no_grad():
            next_q = self.target_net(next_states).max(1)[0]   # 计算下一个状态下所有动作的Q值，并取最大值，得到每个样本的最大Q值

            # 根据贝尔曼方程计算目标Q值
            # 当 dones 为 1 时，表示该回合结束，此时目标Q值仅为当前奖励
            # 当 dones 为 0 时，目标Q值为当前奖励加上折扣因子乘以未来最大Q值
            target_q = rewards + (1 - dones) * self.gamma * next_q

        loss = self.loss_fn(current_q, target_q.unsqueeze(1))  # 计算损失并反向传播
        self.optimizer.zero_grad()  # 清空优化器中累积的梯度，防止梯度累积
        loss.backward()  # 反向传播，计算损失函数关于网络参数的梯度
        self.optimizer.step()  # 使用优化器根据计算得到的梯度更新Q网络的参数
        return loss.item()  # 添加返回loss值

    def sync_target_net(self):
        self.target_net.load_state_dict(self.q_net.state_dict())   # 将Q网络的参数复制到目标网络中，使目标网络的参数与Q网络的参数相同

    def best_action(self, state, CandidateNodesPool):
        state_feature = self._encode_state(state)
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state_feature).unsqueeze(0).to(self.device)
            q_values = self.q_net(state_tensor)
            candidate_indices = [self.candidate_pos_to_index[node] for node in CandidateNodesPool]  # 获取候选节点在动作空间中的索引
            valid_q_values = q_values[0, candidate_indices]  # 从所有动作的 Q 值中选取候选节点对应的 Q 值
            action_idx = torch.argmax(valid_q_values).item()  # 选择 Q 值最大的动作的索引
            return CandidateNodesPool[action_idx]




class ReplayBuffer:
    def __init__(self, capacity):  # capacity是传入的参数，代表缓冲区所能容纳的最大经验数量。
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):  # add方法的作用是把一个经验元组添加到缓冲区里。
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):  # sample方法的用途是从缓冲区里随机采样出一个批次的经验数据。batch_size是传入的参数，代表要采样的经验数量。
        return random.sample(self.buffer, batch_size)

    def __len__(self):  # 返回self.buffer中当前存储的经验数量。
        return len(self.buffer)



In [42]:
import numpy as np

def get_node_hyperedge_avgsize(G_matrix):
 # 获取所有超边的大小
    HyperEdge_size = []
    trans_matrix = G_matrix.T
    for hyperedge in trans_matrix:
        HyperEdge_size.append(np.sum(hyperedge))
# 节点邻居超边平均size
    node_hyperedge_avgsize = np.zeros(G_matrix.shape[0])
    for i in range(G_matrix.shape[0]):
        pos = np.where(G_matrix[i] == 1)[0]
        hyper_num = len(pos)
        for j in pos:
            node_hyperedge_avgsize[i] += HyperEdge_size[j]
        if hyper_num > 0:
            node_hyperedge_avgsize[i] /= hyper_num
        else:
            node_hyperedge_avgsize[i] = 0.0  # 孤立节点直接置为0

    return node_hyperedge_avgsize


def get_avg_HigerOrder_dis(seednodes_pos, G_matrix, data, max_order, alpha):
    avg_HigerOrder_dis_seednodes = np.zeros(G_matrix.shape[0])
    for node in range(G_matrix.shape[0]):
        if node in seednodes_pos:
            continue
        avg_HigerOrder_dis = 0
        for order in range(1, max_order + 1):
            file_path = f'Hyper_Distance/{data}_distance_{order}.npy'
            if not os.path.exists(file_path):
                # 容错处理：如果文件不存在，跳过或打印提醒
                continue
                
            order_matrix = np.load(file_path, allow_pickle=True)

            alpha_ratio = 1 / pow(alpha, order) 

            avg_Order_dis = 0
            for seednode in seednodes_pos:
                if np.isinf(order_matrix[node][seednode]):
                    Order_dis = 0
                else:
                    Order_dis = order_matrix[node][seednode]
                avg_Order_dis += Order_dis
            avg_Order_dis /= len(seednodes_pos)
            avg_HigerOrder_dis += (alpha_ratio * avg_Order_dis)
        avg_HigerOrder_dis_seednodes[node] = avg_HigerOrder_dis
    return avg_HigerOrder_dis_seednodes


def Competition_composite_indicator_new(seednodes_pos, G_matrix, data, select_num, max_order, ind_alpha):
    node_hyperedge_avgsize = get_node_hyperedge_avgsize(G_matrix)
    avg_HigerOrder_dis_seednodes_new = get_avg_HigerOrder_dis(seednodes_pos, G_matrix, data, max_order, ind_alpha)
    composite_indicator = np.array(node_hyperedge_avgsize) * np.array(avg_HigerOrder_dis_seednodes_new)
    ######### 返回的是候选节点的索引 #########
    Candidate_nodes = np.argsort(-composite_indicator)[:select_num]
    return Candidate_nodes






# import os
# from Data import get_Graph
# from SeedNodesGenerator import H_Degree_SG

# data = "Algebra"
# data = "Bars-Rev"
# data = "Restaurants-Rev"
# data = "Geometry"
# data = "Music-Rev"


# if not os.path.exists('../HyperGraphMatrix/' + data + '.npy'):
#     print("saving file...")
#     G_matrix, NodeList = get_Graph(data)
#     np.save('../HyperGraphMatrix/' + data + '.npy', G_matrix)
#     np.save('../HyperGraphMatrix/' + data + '_nodelist.npy', NodeList)
#     print("new data: " + data + " save successfully！")
# else:
#     print("loading file...")
#     G_matrix = np.load('../HyperGraphMatrix/' + data + '.npy', allow_pickle=True)
#     NodeList = np.load('../HyperGraphMatrix/' + data + '_nodelist.npy', allow_pickle=True)
#     NodeList = NodeList.tolist()
#     print("load data: " + data + " successfully！")

# NodeNumber = len(NodeList)
# SeedNodes_number = int(0.01 * NodeNumber)
#
# SNG_mode = 'H_Degree'
# SeedNodes = H_Degree_SG(G_matrix, NodeList, SeedNodes_number)
#
# SeedNodes_pos = []
# for i in SeedNodes:
#     SeedNodes_pos.append(NodeList.index(i))

# node_hyperedge_avgsize, node_neighbor_num, avg_dis_seednodes, avg_HigerOrder_dis_seednodes = node_indicator(SeedNodes_pos, G_matrix)
# print(node_hyperedge_avgsize)
# print(node_neighbor_num)
# print(avg_dis_seednodes)
# print(avg_HigerOrder_dis_seednodes)




In [43]:
import numpy as np
from joblib import Parallel, delayed
from multiprocessing.shared_memory import SharedMemory
from tqdm import tqdm
import os

def HG_Diffusion_SICP(G_matrix, NodeList, SeedNodes, BlockNodes, beta, T, seed=None):
    """
    该函数实现了HG_Diffusion_SICP优化的扩散过程。

    参数:
    G_matrix (numpy.ndarray): 超图的邻接矩阵，形状为(num_nodes, num_hyperedges)
    NodeList (list): 节点名称列表
    SeedNodes (list): 初始谣言节点名称列表
    BlockNodes (list): 初始阻塞节点名称列表
    beta (float): 激活概率
    T (int): 扩散的时间步数

    返回:
    tuple: 包含节点总数、未激活节点数、谣言节点数和阻塞节点数的元组
    """

    # 添加seed参数
    if seed is not None:
        np.random.seed(seed)  # 固定随机种子
    # 预处理：预计算所有超边的节点列表和节点的超边列表
    # 获取节点数量和超边数量
    num_nodes, num_hyperedges = G_matrix.shape
    # 对于每个超边，找出与之相连的节点的索引
    hyperedge_nodes = [np.where(G_matrix[:, he] == 1)[0] for he in range(num_hyperedges)]
    # 对于每个节点，找出与之相连的超边的索引
    node_hyperedges = [np.where(G_matrix[node_idx] == 1)[0] for node_idx in range(num_nodes)]

    # 预生成所有节点的超边选择（形状：[num_nodes, T]）
    # 初始化超边选择数组，初始值为 -1
    hyperedge_choices = np.full((num_nodes, T), -1, dtype=int)
    # 遍历每个节点
    for idx in range(num_nodes):
        # 获取该节点相连的超边索引
        hes = node_hyperedges[idx]
        # 如果该节点有相连的超边
        if len(hes) > 0:
            rng = np.random.RandomState(seed) if seed is not None else np.random
            # 为该节点在每个时间步随机选择一个超边
            hyperedge_choices[idx] = np.random.choice(hes, size=T)

    # 转换节点名为索引
    # 将阻塞节点名称转换为对应的索引
    BlockNodes_index = [NodeList.index(node) for node in BlockNodes]
    # 将谣言节点名称转换为对应的索引
    RumorNodes_index = [NodeList.index(node) for node in SeedNodes]

    # 初始化状态（1: 阻塞, -1: 谣言, 0: 未激活）
    # 初始化节点状态数组，初始值为 0
    StateList = np.zeros(num_nodes, dtype=np.int8)
    # 将阻塞节点的状态设置为 1
    StateList[BlockNodes_index] = 1
    # 将谣言节点的状态设置为 -1
    StateList[RumorNodes_index] = -1

    # 使用集合存储激活节点
    # 将阻塞节点索引转换为集合
    BlockNodes_set = set(BlockNodes_index)
    # 将谣言节点索引转换为集合
    RumorNodes_set = set(RumorNodes_index)

    # 进行 T 个时间步的扩散
    for t in range(T):
        # 处理阻塞节点传播
        # 如果有阻塞节点
        if BlockNodes_set:
            # 将阻塞节点集合转换为数组
            active_block = np.array(list(BlockNodes_set))
            # 获取当前时间步这些阻塞节点选择的超边
            current_hyper = hyperedge_choices[active_block, t]
            # 筛选出选择了有效超边的节点
            valid_mask = current_hyper != -1
            valid_actives = active_block[valid_mask]
            valid_hes = current_hyper[valid_mask]

            # 如果有有效激活的阻塞节点
            if valid_actives.size > 0:
                # 获取这些有效超边所连接的所有邻居节点
                all_neighbors = np.concatenate([hyperedge_nodes[he] for he in valid_hes])
                # 筛选出未激活的邻居节点
                unactivated = all_neighbors[StateList[all_neighbors] == 0]
                # 如果有未激活的邻居节点
                if unactivated.size > 0:
                    # 根据激活概率随机选择要激活的节点
                    activated = unactivated[np.random.random(unactivated.size) <= beta]
                    # 将这些激活的节点状态设置为 1
                    StateList[activated] = 1
                    # 将这些激活的节点添加到阻塞节点集合中
                    BlockNodes_set.update(activated.tolist())

        # 处理谣言节点传播
        # 如果有谣言节点
        if RumorNodes_set:
            # 将谣言节点集合转换为数组
            active_rumor = np.array(list(RumorNodes_set))
            # 获取当前时间步这些谣言节点选择的超边
            current_hyper = hyperedge_choices[active_rumor, t]
            # 筛选出选择了有效超边的节点
            valid_mask = current_hyper != -1
            valid_actives = active_rumor[valid_mask]
            valid_hes = current_hyper[valid_mask]

            # 如果有有效激活的谣言节点
            if valid_actives.size > 0:
                # 获取这些有效超边所连接的所有邻居节点
                all_neighbors = np.concatenate([hyperedge_nodes[he] for he in valid_hes])
                # 筛选出未激活的邻居节点
                unactivated = all_neighbors[StateList[all_neighbors] == 0]
                # 如果有未激活的邻居节点
                if unactivated.size > 0:
                    # 根据激活概率随机选择要激活的节点
                    activated = unactivated[np.random.random(unactivated.size) <= beta]
                    # 将这些激活的节点状态设置为 -1
                    StateList[activated] = -1
                    # 将这些激活的节点添加到谣言节点集合中
                    RumorNodes_set.update(activated.tolist())

    # 统计结果
    # 统计未激活节点的数量
    unactivated_number = np.sum(StateList == 0)
    # 返回节点总数、未激活节点数、谣言节点数和阻塞节点数
    return len(NodeList), unactivated_number, len(RumorNodes_set), len(BlockNodes_set)



def HG_Diffusion_SICP_pos(G_matrix, NodeList, SeedNodes, BlockNodes_pos, beta, T, seed=None):
    """
    该函数实现了HG_Diffusion_SICP_pos优化的扩散过程。

    参数:
    G_matrix (numpy.ndarray): 超图的邻接矩阵，形状为(num_nodes, num_hyperedges)
    NodeList (list): 节点名称列表
    SeedNodes (list): 初始谣言节点名称列表
    BlockNodes_pos (list): 初始阻塞节点的索引列表
    beta (float): 激活概率
    T (int): 扩散的时间步数

    返回:
    tuple: 包含节点总数、未激活节点数、谣言节点数和阻塞节点数的元组
    """

    # 添加seed参数
    if seed is not None:
        np.random.seed(seed)  # 固定随机种子
    # 预处理与上述函数相同，仅BlockNodes_index处理不同
    # 获取节点数量和超边数量
    num_nodes, num_hyperedges = G_matrix.shape
    # 对于每个超边，找出与之相连的节点的索引
    hyperedge_nodes = [np.where(G_matrix[:, he] == 1)[0] for he in range(num_hyperedges)]
    # 对于每个节点，找出与之相连的超边的索引
    node_hyperedges = [np.where(G_matrix[node_idx] == 1)[0] for node_idx in range(num_nodes)]

    # 预生成所有节点的超边选择（形状：[num_nodes, T]）
    # 初始化超边选择数组，初始值为 -1
    hyperedge_choices = np.full((num_nodes, T), -1, dtype=int)
    # 遍历每个节点
    for idx in range(num_nodes):
        # 获取该节点相连的超边索引
        hes = node_hyperedges[idx]
        # 如果该节点有相连的超边
        if len(hes) > 0:
            # 使用固定种子生成随机选择
            rng = np.random.RandomState(seed) if seed is not None else np.random
            # 为该节点在每个时间步随机选择一个超边
            hyperedge_choices[idx] = np.random.choice(hes, size=T)

    # 这里直接使用传入的阻塞节点索引
    BlockNodes_index = BlockNodes_pos
    # 将谣言节点名称转换为对应的索引
    RumorNodes_index = [NodeList.index(node) for node in SeedNodes]

    # 初始化状态（1: 阻塞, -1: 谣言, 0: 未激活）
    # 初始化节点状态数组，初始值为 0
    StateList = np.zeros(num_nodes, dtype=np.int8)
    # 将阻塞节点的状态设置为 1
    StateList[BlockNodes_index] = 1
    # 将谣言节点的状态设置为 -1
    StateList[RumorNodes_index] = -1

    # 使用集合存储激活节点
    # 将阻塞节点索引转换为集合
    BlockNodes_set = set(BlockNodes_index)
    # 将谣言节点索引转换为集合
    RumorNodes_set = set(RumorNodes_index)

    # 进行 T 个时间步的扩散
    for t in range(T):
        # 处理阻塞节点传播
        # 如果有阻塞节点
        if BlockNodes_set:
            # 将阻塞节点集合转换为数组
            active_block = np.array(list(BlockNodes_set))
            # 获取当前时间步这些阻塞节点选择的超边
            current_hyper = hyperedge_choices[active_block, t]
            # 筛选出选择了有效超边的节点
            valid_mask = current_hyper != -1
            valid_actives = active_block[valid_mask]
            valid_hes = current_hyper[valid_mask]

            # 如果有有效激活的阻塞节点
            if valid_actives.size > 0:
                # 获取这些有效超边所连接的所有邻居节点
                all_neighbors = np.concatenate([hyperedge_nodes[he] for he in valid_hes])
                # 筛选出未激活的邻居节点
                unactivated = all_neighbors[StateList[all_neighbors] == 0]
                # 如果有未激活的邻居节点
                if unactivated.size > 0:
                    # 根据激活概率随机选择要激活的节点
                    activated = unactivated[np.random.random(unactivated.size) <= beta]
                    # 将这些激活的节点状态设置为 1
                    StateList[activated] = 1
                    # 将这些激活的节点添加到阻塞节点集合中
                    BlockNodes_set.update(activated.tolist())

        # 处理谣言节点传播
        # 如果有谣言节点
        if RumorNodes_set:
            # 将谣言节点集合转换为数组
            active_rumor = np.array(list(RumorNodes_set))
            # 获取当前时间步这些谣言节点选择的超边
            current_hyper = hyperedge_choices[active_rumor, t]
            # 筛选出选择了有效超边的节点
            valid_mask = current_hyper != -1
            valid_actives = active_rumor[valid_mask]
            valid_hes = current_hyper[valid_mask]

            # 如果有有效激活的谣言节点
            if valid_actives.size > 0:
                # 获取这些有效超边所连接的所有邻居节点
                all_neighbors = np.concatenate([hyperedge_nodes[he] for he in valid_hes])
                # 筛选出未激活的邻居节点
                unactivated = all_neighbors[StateList[all_neighbors] == 0]
                # 如果有未激活的邻居节点
                if unactivated.size > 0:
                    # 根据激活概率随机选择要激活的节点
                    activated = unactivated[np.random.random(unactivated.size) <= beta]
                    # 将这些激活的节点状态设置为 -1
                    StateList[activated] = -1
                    # 将这些激活的节点添加到谣言节点集合中
                    RumorNodes_set.update(activated.tolist())

    # 统计结果
    # 统计未激活节点的数量
    unactivated_number = np.sum(StateList == 0)
    # 返回节点总数、未激活节点数、谣言节点数和阻塞节点数
    return len(NodeList), unactivated_number, len(RumorNodes_set), len(BlockNodes_set)


def _run_simulation(diffusion_func, task_args):
    """全局辅助函数，处理共享内存并调用扩散函数（带强力防崩溃保护）"""
    if len(task_args) == 9:
        shm_name, dtype, shape, NodeList, SeedNodes, BlockNodes, beta, T, seed = task_args
    else:
        shm_name, dtype, shape, NodeList, SeedNodes, BlockNodes, beta, T = task_args
        seed = None

    shm = None
    G_matrix = None  # 提前声明，防止 finally 报错
    try:
        shm = SharedMemory(name=shm_name)
        G_matrix = np.ndarray(shape, dtype=dtype, buffer=shm.buf)
        res = diffusion_func(G_matrix, NodeList, SeedNodes, BlockNodes, beta, T, seed=seed)
        return res
    finally:
        if G_matrix is not None:
            del G_matrix  # 显式删除 numpy 数组对象
        if shm is not None:
            shm.close()  # 关闭共享内存连接

        # 强制触发 Python 垃圾回收器，清理底层 C 语言层面的 mmap 句柄
        import gc
        gc.collect()


def Simulations(simulation_num, diffusion_func, shm_name, dtype, shape, *args):
    """并行执行多次扩散模拟"""
    base_seed = 42
    seeds = [base_seed + i for i in range(simulation_num)]
    tasks = [(shm_name, dtype, shape) + args + (seed,) for seed in seeds]

    # 【关键修复3】：绝对不能用 n_jobs=-1！将其固定为 4。
    # 如果您的电脑依然报 WinError 1455，请将这里的 4 改为 2。
    results = Parallel(n_jobs=4, backend="loky")(
        delayed(_run_simulation)(diffusion_func, task)
        for task in tasks
    )

    if results and isinstance(results[0], tuple):
        blocked_counts = [result[3] for result in results]
    else:
        blocked_counts = results

    return np.mean(blocked_counts)



def Simulations_tqdm(simulation_num, diffusion_func, shm_name, dtype, shape, *args):
    """
    并行执行多次扩散模拟（使用 joblib + 共享内存优化）
    """
    # 生成任务列表
    tasks = [(shm_name, dtype, shape) + args for _ in range(simulation_num)]

    # 使用 joblib 并行执行
    results = Parallel(n_jobs=-1)(
        delayed(_run_simulation)(diffusion_func, task)
        for task in tqdm(tasks, desc="模拟进度", unit="次")
    )

    # 计算平均阻塞节点数
    blocked_counts = [result[3] for result in results]
    return np.mean(blocked_counts)



def HG_Diffusion_SICP_track(G_matrix, NodeList, SeedNodes, BlockNodes, beta, T, interval):
    """
    该函数实现了HG_Diffusion_SICP_pos优化的扩散过程，并跟踪每个间隔时间的BlockNodes数量。

    参数:
    G_matrix (numpy.ndarray): 超图的邻接矩阵，形状为(num_nodes, num_hyperedges)
    NodeList (list): 节点名称列表
    SeedNodes (list): 初始谣言节点名称列表
    BlockNodes_pos (list): 初始阻塞节点的索引列表
    beta (float): 激活概率
    T (int): 扩散的时间步数
    interval (int): 记录BlockNodes数量的时间间隔

    返回:
    tuple: 包含节点总数、未激活节点数、谣言节点数、阻塞节点数和各时间间隔BlockNodes数量列表的元组
    """
    num_nodes, num_hyperedges = G_matrix.shape
    hyperedge_nodes = [np.where(G_matrix[:, he] == 1)[0] for he in range(num_hyperedges)]
    node_hyperedges = [np.where(G_matrix[node_idx] == 1)[0] for node_idx in range(num_nodes)]

    hyperedge_choices = np.full((num_nodes, T), -1, dtype=int)
    for idx in range(num_nodes):
        hes = node_hyperedges[idx]
        if len(hes) > 0:
            hyperedge_choices[idx] = np.random.choice(hes, size=T)


    # 转换节点名为索引
    # 将阻塞节点名称转换为对应的索引
    BlockNodes_index = [NodeList.index(node) for node in BlockNodes]
    # 将谣言节点名称转换为对应的索引
    RumorNodes_index = [NodeList.index(node) for node in SeedNodes]

    StateList = np.zeros(num_nodes, dtype=np.int8)
    StateList[BlockNodes_index] = 1
    StateList[RumorNodes_index] = -1

    BlockNodes_set = set(BlockNodes_index)
    RumorNodes_set = set(RumorNodes_index)

    track_counts = [len(BlockNodes_set)]  # 初始状态

    for t in range(T):
        # 处理阻塞节点传播
        if BlockNodes_set:
            active_block = np.array(list(BlockNodes_set))
            current_hyper = hyperedge_choices[active_block, t]
            valid_mask = current_hyper != -1
            valid_actives = active_block[valid_mask]
            valid_hes = current_hyper[valid_mask]

            if valid_actives.size > 0:
                all_neighbors = np.concatenate([hyperedge_nodes[he] for he in valid_hes])
                unactivated = all_neighbors[StateList[all_neighbors] == 0]
                if unactivated.size > 0:
                    activated = unactivated[np.random.random(unactivated.size) <= beta]
                    StateList[activated] = 1
                    BlockNodes_set.update(activated.tolist())

        # 处理谣言节点传播
        if RumorNodes_set:
            active_rumor = np.array(list(RumorNodes_set))
            current_hyper = hyperedge_choices[active_rumor, t]
            valid_mask = current_hyper != -1
            valid_actives = active_rumor[valid_mask]
            valid_hes = current_hyper[valid_mask]

            if valid_actives.size > 0:
                all_neighbors = np.concatenate([hyperedge_nodes[he] for he in valid_hes])
                unactivated = all_neighbors[StateList[all_neighbors] == 0]
                if unactivated.size > 0:
                    activated = unactivated[np.random.random(unactivated.size) <= beta]
                    StateList[activated] = -1
                    RumorNodes_set.update(activated.tolist())

        # 记录间隔时间点的BlockNodes数量
        current_time = t + 1
        if current_time % interval == 0 or current_time == T:
            track_counts.append(len(BlockNodes_set))

    unactivated_number = np.sum(StateList == 0)
    return len(NodeList), unactivated_number, len(RumorNodes_set), track_counts



def HG_Diffusion_SICP_pos_track(G_matrix, NodeList, SeedNodes, BlockNodes_pos, beta, T, interval):
    """
    该函数实现了HG_Diffusion_SICP_pos优化的扩散过程，并跟踪每个间隔时间的BlockNodes数量。

    参数:
    G_matrix (numpy.ndarray): 超图的邻接矩阵，形状为(num_nodes, num_hyperedges)
    NodeList (list): 节点名称列表
    SeedNodes (list): 初始谣言节点名称列表
    BlockNodes_pos (list): 初始阻塞节点的索引列表
    beta (float): 激活概率
    T (int): 扩散的时间步数
    interval (int): 记录BlockNodes数量的时间间隔

    返回:
    tuple: 包含节点总数、未激活节点数、谣言节点数、阻塞节点数和各时间间隔BlockNodes数量列表的元组
    """
    num_nodes, num_hyperedges = G_matrix.shape
    hyperedge_nodes = [np.where(G_matrix[:, he] == 1)[0] for he in range(num_hyperedges)]
    node_hyperedges = [np.where(G_matrix[node_idx] == 1)[0] for node_idx in range(num_nodes)]

    hyperedge_choices = np.full((num_nodes, T), -1, dtype=int)
    for idx in range(num_nodes):
        hes = node_hyperedges[idx]
        if len(hes) > 0:
            hyperedge_choices[idx] = np.random.choice(hes, size=T)

    BlockNodes_index = BlockNodes_pos
    RumorNodes_index = [NodeList.index(node) for node in SeedNodes]

    StateList = np.zeros(num_nodes, dtype=np.int8)
    StateList[BlockNodes_index] = 1
    StateList[RumorNodes_index] = -1

    BlockNodes_set = set(BlockNodes_index)
    RumorNodes_set = set(RumorNodes_index)

    track_counts = [len(BlockNodes_set)]  # 初始状态

    for t in range(T):
        # 处理阻塞节点传播
        if BlockNodes_set:
            active_block = np.array(list(BlockNodes_set))
            current_hyper = hyperedge_choices[active_block, t]
            valid_mask = current_hyper != -1
            valid_actives = active_block[valid_mask]
            valid_hes = current_hyper[valid_mask]

            if valid_actives.size > 0:
                all_neighbors = np.concatenate([hyperedge_nodes[he] for he in valid_hes])
                unactivated = all_neighbors[StateList[all_neighbors] == 0]
                if unactivated.size > 0:
                    activated = unactivated[np.random.random(unactivated.size) <= beta]
                    StateList[activated] = 1
                    BlockNodes_set.update(activated.tolist())

        # 处理谣言节点传播
        if RumorNodes_set:
            active_rumor = np.array(list(RumorNodes_set))
            current_hyper = hyperedge_choices[active_rumor, t]
            valid_mask = current_hyper != -1
            valid_actives = active_rumor[valid_mask]
            valid_hes = current_hyper[valid_mask]

            if valid_actives.size > 0:
                all_neighbors = np.concatenate([hyperedge_nodes[he] for he in valid_hes])
                unactivated = all_neighbors[StateList[all_neighbors] == 0]
                if unactivated.size > 0:
                    activated = unactivated[np.random.random(unactivated.size) <= beta]
                    StateList[activated] = -1
                    RumorNodes_set.update(activated.tolist())

        # 记录间隔时间点的BlockNodes数量
        current_time = t + 1
        if current_time % interval == 0 or current_time == T:
            track_counts.append(len(BlockNodes_set))

    unactivated_number = np.sum(StateList == 0)
    return len(NodeList), unactivated_number, len(RumorNodes_set), track_counts



def _run_simulation_track(diffusion_func, task_args):
    """处理共享内存并调用跟踪型扩散函数的辅助函数"""
    shm_name, dtype, shape, NodeList, SeedNodes, BlockNodes, beta, T, interval = task_args
    try:
        shm = SharedMemory(name=shm_name)
        G_matrix = np.ndarray(shape, dtype=dtype, buffer=shm.buf)
        return diffusion_func(G_matrix, NodeList, SeedNodes, BlockNodes, beta, T, interval)
    finally:
        shm.close()


def Simulations_track(simulation_num, diffusion_func, shm_name, dtype, shape, *args):
    """
    并行执行多次跟踪型扩散模拟，返回各时间间隔的平均阻塞节点数量

    参数:
    simulation_num (int): 模拟次数
    diffusion_func (callable): 跟踪型扩散函数（如HG_Diffusion_SICP_pos_track）
    shm_name (str): 共享内存名称
    dtype: 矩阵数据类型
    shape: 矩阵形状
    *args: 扩散函数参数 (NodeList, SeedNodes, BlockNodes, beta, T, interval)

    返回:
    tuple: (平均最终阻塞数, 各时间点平均阻塞数列表)
    """
    tasks = [(shm_name, dtype, shape) + args for _ in range(simulation_num)]

    results = Parallel(n_jobs=-1)(
        delayed(_run_simulation_track)(diffusion_func, task)
        for task in tasks
    )

    # 提取各时间点数据
    time_series = [result[3] for result in results]

    # 计算各时间点平均值
    avg_series = np.mean(time_series, axis=0).tolist()

    return avg_series



def Simulations_track_tqdm(simulation_num, diffusion_func, shm_name, dtype, shape, *args):
    """
    并行执行多次跟踪型扩散模拟，返回各时间间隔的平均阻塞节点数量

    参数:
    simulation_num (int): 模拟次数
    diffusion_func (callable): 跟踪型扩散函数（如HG_Diffusion_SICP_pos_track）
    shm_name (str): 共享内存名称
    dtype: 矩阵数据类型
    shape: 矩阵形状
    *args: 扩散函数参数 (NodeList, SeedNodes, BlockNodes, beta, T, interval)

    返回:
    tuple: (平均最终阻塞数, 各时间点平均阻塞数列表)
    """
    tasks = [(shm_name, dtype, shape) + args for _ in range(simulation_num)]

    results = Parallel(n_jobs=-1)(
        delayed(_run_simulation_track)(diffusion_func, task)
        for task in tqdm(tasks, desc="跟踪模拟进度", unit="次")
    )

    # 提取各时间点数据
    time_series = [result[3] for result in results]

    # 计算各时间点平均值
    avg_series = np.mean(time_series, axis=0).tolist()

    return avg_series



In [44]:
import numpy as np
from tqdm import trange
from tqdm import tqdm
from multiprocessing.shared_memory import SharedMemory

# ==========================================
# ⚠️ 已经为您删除了这里会导致报错的 from Diffusion_In_HyperGraph import ...
# 因为这些函数在前面的 Cell 里已经定义并加载到内存了
# ==========================================

# 用于根据超图的超度选择竞争节点，返回节点名称,返回list
def H_Degree(G_matrix, NodeList, SeedNodes, CompeteNodeNumber):
    # 计算每个节点的超度，即 G_matrix 每一行元素的和
    hyper_degree = np.sum(G_matrix, axis=1)
    # 创建一个布尔掩码，用于标记不在种子节点列表中的节点
    candidate_mask = ~np.isin(NodeList, SeedNodes)
    # 根据掩码筛选出候选节点
    candidates = np.array(NodeList)[candidate_mask]
    # 对候选节点的超度进行降序排序，并获取排序后的索引
    sorted_indices = np.argsort(-hyper_degree[candidate_mask])
    # 选取超度最大的 CompeteNodeNumber 个节点作为竞争节点，并转换为列表返回
    return candidates[sorted_indices[:CompeteNodeNumber]].tolist()

# 用于根据超图中节点的度选择竞争节点，返回节点名称,返回list
def Degree(G_matrix,NodeList,SeedNodes,CompeteNodeNumber):
    DegreeList = []
    for i in range(len(G_matrix)):
        HyperEdge_index = np.where(G_matrix[i] == 1)[0]
        Neighbors_index = []
        for j in HyperEdge_index:
            Neighbors_index += list(np.where(G_matrix[:,j] == 1)[0])
        Neighbors_index = np.unique(Neighbors_index)
        DegreeList.append(len(Neighbors_index) - 1)

    HyperEdgeList_sort_index = np.argsort(-np.array(DegreeList))
    number = 0
    CompeteNodes = []
    for i in HyperEdgeList_sort_index:
        if number < CompeteNodeNumber:
            if NodeList[i] not in SeedNodes:
                CompeteNodes.append(NodeList[i])
                number += 1
        else:
            break
    return CompeteNodes

# 输入的是种子节点序号，候选节点索引，返回的是节点索引,返回list
def Indicator_selection_Greedy(G_matrix, NodeList, SeedNodes, CandidateNodes_pos:list, CompeteNodeNumber, greedy_times, beta, T):
    greedy_pos_list = []
    pool = CandidateNodes_pos.copy()

    # 开辟共享内存
    shm = SharedMemory(create=True, size=G_matrix.nbytes)
    shm_np = np.ndarray(G_matrix.shape, dtype=G_matrix.dtype, buffer=shm.buf)
    np.copyto(shm_np, G_matrix)

    for _ in trange(CompeteNodeNumber, desc="indicator_selection_greedy"):
        max_inf = 0
        max_node_pos = 0

        for node_pos in pool:
            greedy_pos_list_copy = greedy_pos_list.copy()
            result_inf = Simulations(
                greedy_times,
                HG_Diffusion_SICP_pos,
                shm.name,  # 共享内存名称
                G_matrix.dtype,  # 数据类型
                G_matrix.shape,  # 矩阵形状
                NodeList,
                SeedNodes,
                greedy_pos_list_copy + [node_pos],
                beta,
                T
            )
            if result_inf > max_inf:
                max_inf = result_inf
                max_node_pos = node_pos

        pool.remove(max_node_pos)
        greedy_pos_list.append(max_node_pos)

    # 关闭共享内存
    shm.close()
    shm.unlink()

    return greedy_pos_list

def HADP(G_matrix, NodeList, SeedNodes, CompeteNodeNumber):
    G_matrix_copy = G_matrix.copy()

    for i in SeedNodes:
        idx = NodeList.index(i)
        for j in range(len(G_matrix_copy[idx])):
            G_matrix_copy[idx][j] = 0

    HADP_list_idx = []

    for i in trange(CompeteNodeNumber):
        Degree_max = 0
        Degree_max_idx = -1
        for j in range(len(G_matrix_copy)):
            HyperEdge_index = np.where(G_matrix_copy[j] == 1)[0]
            Degree_list = []
            for k in HyperEdge_index:
                Degree_list += list(np.where(G_matrix_copy[:, k] == 1)[0])
            Degree_list = np.unique(Degree_list)
            if len(Degree_list) > Degree_max:
                Degree_max = len(Degree_list)
                Degree_max_idx = j
        HADP_list_idx.append(Degree_max_idx)
        for j in range(len(G_matrix_copy[Degree_max_idx])):
            G_matrix_copy[Degree_max_idx][j] = 0

    CompeteNodes = []
    for i in HADP_list_idx:
        CompeteNodes.append(NodeList[i])

    return CompeteNodes

def CI(G_matrix, NodeList, SeedNodes, CompeteNodeNumber):
    CI_list = []
    G_matrix_copy = G_matrix.copy()

    for i in trange(len(G_matrix)):
        HyperEdge_index = np.where(G_matrix_copy[i] == 1)[0]
        ci = len(HyperEdge_index) - 1
        ball_nodes_idx = []
        for j in HyperEdge_index:
            ball_nodes_idx += list(np.where(G_matrix_copy[:, j] == 1)[0])
        ball_nodes_idx = np.unique(ball_nodes_idx)
        res = 0
        for j in ball_nodes_idx:
            res += (len(np.where(G_matrix_copy[j] == 1)[0]) - 1)
        ci += res
        CI_list.append(ci)

    CI_sort_idx = np.argsort(- np.array(CI_list))

    CompeteNodes = []
    idx = 0
    while True:
        if len(CompeteNodes) >= CompeteNodeNumber:
            break

        if NodeList[CI_sort_idx[idx]] not in SeedNodes:
            CompeteNodes.append(NodeList[CI_sort_idx[idx]])
        idx += 1

    return CompeteNodes

In [45]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
import torch.nn.functional as F


class Structure2Vec(nn.Module):
    def __init__(self, embed_dim, nodes_number, num_iterations):
        super().__init__()
        self.embed_dim = embed_dim
        self.theta1 = nn.Linear(2, embed_dim)  # 输入维度为2（己方/对手标签）
        self.theta2 = nn.Linear(embed_dim, embed_dim)
        self.theta3 = nn.Linear(embed_dim, embed_dim)
        self.theta4 = nn.Linear(nodes_number, embed_dim)  # 边权重转换
        self.num_iterations = num_iterations

        self.norm1 = nn.LayerNorm(embed_dim)  # 归一化层
        self.norm2 = nn.LayerNorm(embed_dim)  # 归一化层
        self.norm3 = nn.LayerNorm(embed_dim)  # 归一化层

    def forward(self, x, adj_matrix, edge_weights):
        # x: 节点标签 [batch_size, num_nodes, 2]
        # adj_matrix: 邻接矩阵 [num_nodes, num_nodes]
        # edge_weights: 边权重 [num_nodes, num_nodes]
        mu = torch.zeros(x.size(0), x.size(1), self.embed_dim).to(x.device)
        for _ in range(self.num_iterations):
            theta1_out = self.theta1(x)  # [batch_size, num_nodes, embed_dim]
            theta4_out = self.theta4(edge_weights)  # [num_nodes, embed_dim]
            adj_mul_mu = torch.matmul(adj_matrix, mu)  # [batch_size, num_nodes, embed_dim]
            adj_mul_theta4 = torch.matmul(adj_matrix, F.relu(theta4_out))  # [num_nodes, embed_dim]

            # 添加归一化操作
            theta1_out = self.norm1(theta1_out)  # 归一化操作
            adj_mul_mu = self.norm2(adj_mul_mu)  # 归一化操作
            adj_mul_theta4 = self.norm3(adj_mul_theta4)  # 归一化操作

            # 组合并激活
            mu_new = F.relu(theta1_out + self.theta2(adj_mul_mu) + self.theta3(adj_mul_theta4))
            mu = mu_new
        return mu


class QNetwork(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.theta5 = nn.Linear(2 * embed_dim, 1)  # 拼接全局和节点嵌入
        self.theta6 = nn.Linear(embed_dim, embed_dim)
        self.theta7 = nn.Linear(embed_dim, embed_dim)

        self.norm6 = nn.LayerNorm(embed_dim)  # 归一化层
        self.norm7 = nn.LayerNorm(embed_dim)  # 归一化层

    def forward(self, node_embs):
        # global_emb: 全局池化嵌入 [batch_size, embed_dim]
        # node_embs: 节点嵌入 [batch_size, num_nodes, embed_dim]
        global_emb = torch.mean(node_embs, dim=1)
        global_proj = self.theta6(global_emb.unsqueeze(1))  # [batch_size, 1, embed_dim]
        node_proj = self.theta7(node_embs)                  # [batch_size, num_nodes, embed_dim]

        # 添加归一化操作
        global_proj = self.norm6(global_proj)  # 归一化操作
        node_proj = self.norm7(node_proj)  # 归一化操作

        combined = torch.cat([global_proj.expand(-1, node_embs.size(1), -1), node_proj], dim=-1)
        q_values = self.theta5(F.relu(combined)).squeeze(-1)  # [batch_size, num_nodes]

        return q_values


def state_to_tensor(state, nodes_number, device):
    tensor = torch.zeros((1, nodes_number, 2), device=device, dtype=torch.float32)
    for idx in state:
        tensor[0, idx, 0] = 1.0
    return tensor


class Compete_S2V_DQN:
    def __init__(self, epsilon, alpha, gamma, adj_matrix, edge_weights, nodes_number, candidate_pos, device, embedding_dim):
        # 【修改点1】：S2V 网络放入指定的 device (解决设备不匹配报错)
        self.s2v = Structure2Vec(embedding_dim, nodes_number, num_iterations=32).to(device)
        
        # Q 网络和目标网络
        self.q_net = QNetwork(embedding_dim).to(device)
        self.target_q_net = QNetwork(embedding_dim).to(device)
        
        # 【修改点2】：将 q_net 和 s2v 的参数合在一起交给优化器 (解决 S2V 权重不更新的问题)
        self.optimizer = torch.optim.Adam(
            list(self.q_net.parameters()) + list(self.s2v.parameters()), 
            lr=alpha
        )
        self.epsilon = epsilon
        self.gamma = gamma
        self.adj_matrix = torch.tensor(adj_matrix, dtype=torch.float32).to(device)
        self.edge_weights = torch.tensor(edge_weights, dtype=torch.float32).to(device)
        self.nodes_number = nodes_number
        self.candidate_pos = candidate_pos
        self.device = device

    def take_action(self, state, candidate_nodes):
        # ε-贪心策略
        if np.random.rand() < self.epsilon:
            return np.random.choice(candidate_nodes)
        else:
            x = state_to_tensor(state, self.nodes_number, self.device)
            with torch.no_grad():
                node_embs = self.s2v(x, self.adj_matrix, self.edge_weights)  # 权重都为1
                q_values = self.q_net(node_embs)
                q_values = q_values.squeeze(0)
            return candidate_nodes[torch.argmax(q_values[candidate_nodes]).item()]

    def best_action(self, state, candidate_nodes):
        x = state_to_tensor(state, self.nodes_number, self.device)
        # 贪婪策略（推理阶段使用）
        with torch.no_grad():
            node_embs = self.s2v(x, self.adj_matrix, self.edge_weights)
            q_values = self.q_net(node_embs)

        q_values = q_values[0]
        # 从候选节点中选择具有最大 q 值的节点
        candidate_q_values = q_values[candidate_nodes]
        best_index = torch.argmax(candidate_q_values).item()
        return candidate_nodes[best_index]

    def sync_target_net(self):
        self.target_q_net.load_state_dict(self.q_net.state_dict())

    def update(self, batch):
        # 初始化空列表来存储每个样本的状态、动作、奖励、下一个状态和是否结束标志
        states = []
        actions = []
        rewards = []
        next_states = []
        dones = []

        # 遍历批次中的每个样本
        for state, action, reward, next_state, done in batch:
            states.append(state)
            actions.append(action)
            rewards.append(reward)
            next_states.append(next_state)
            dones.append(done)

        states_tensor = torch.cat([state_to_tensor(state, self.nodes_number, self.device) for state in states], dim=0)
        actions = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)  # 将奖励数据转换为FloatTensor类型的张量，并移动到指定设备上
        next_states_tensor = torch.cat([state_to_tensor(next_state, self.nodes_number, self.device) for next_state in next_states], dim=0)
        dones = torch.FloatTensor(dones).to(self.device)  # 将是否结束的标志数据转换为FloatTensor类型的张量，并移动到指定设备上

        next_states_embs = self.s2v(next_states_tensor, self.adj_matrix, self.edge_weights)
        states_embs = self.s2v(states_tensor, self.adj_matrix, self.edge_weights)

        # 【修改点3】：加入 Mask 掩码屏蔽非法节点，加入 (1 - dones) 截断终止状态收益
        with torch.no_grad():
            next_q_values = self.target_q_net(next_states_embs)
            mask = torch.full_like(next_q_values, float('-inf'))
            for i, n_state in enumerate(next_states):
                available_nodes = list(set(self.candidate_pos) - set(n_state))
                if len(available_nodes) > 0:
                    mask[i, available_nodes] = 0.0
            
            max_next_q = (next_q_values + mask).max(1)[0]
            target_q = rewards + (1 - dones) * self.gamma * max_next_q

        q_values = self.q_net(states_embs)
        current_q = q_values.gather(1, actions).squeeze(1)  # 确保 current_q 的尺寸为 [batch_size]

        # 确保 target_q 和 current_q 的尺寸一致
        assert current_q.shape == target_q.shape, f"current_q shape: {current_q.shape}, target_q shape: {target_q.shape}"

        # 计算损失并更新
        loss = F.mse_loss(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        
        # 【修改点4】：梯度裁剪加入 S2V 参数
        torch.nn.utils.clip_grad_norm_(
            list(self.q_net.parameters()) + list(self.s2v.parameters()), 
            max_norm=1.0
        )
        self.optimizer.step()
        return loss.item()


class S2VReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

In [39]:
import numpy as np
import os
import torch
import random
import time
from multiprocessing.shared_memory import SharedMemory
from tqdm import trange

# 1. 自动适配当前路径
CURRENT_PATH = os.getcwd() 
# 如果你的 Hyper_Distance 文件夹就在当前目录下，请确保下面的函数被更新
# 或者在运行前手动修改 Indicator.py 里的路径

def train_and_save_CSIQ(data_name, beta=0.03, T=30):
    start = time.time()

    # 基础超参数
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    CandidateNodes_Ratio = 0.2
    Indicator_Order = 10
    Indicator_Alpha = 0.5
    num_episodes = 600
    alpha = 0.03
    gamma = 0.9
    epsilon_start = 0.5
    epsilon_end = 0.01
    epsilon_decay = (epsilon_start - epsilon_end) / num_episodes
    simulation_num = 50
    batch_size = 64
    buffer_capacity = 30000
    target_update_freq = 100

    # 2. 数据加载路径修改：直接在当前目录下寻找
    m_p = os.path.join(CURRENT_PATH, f"{data_name}.npy")
    n_p = os.path.join(CURRENT_PATH, f"{data_name}_nodelist.npy")

    if not os.path.exists(m_p):
        print(f"❌ 找不到数据文件: {m_p}")
        return

    G_matrix = np.load(m_p, allow_pickle=True)
    NodeList = np.load(n_p, allow_pickle=True).tolist()

    num_v = int(0.01 * len(NodeList))
    state_dim = 2 * num_v

    # 3. 初始化种子节点
    SeedNodes = H_Degree_SG(G_matrix, NodeList, num_v)
    SeedNodes_pos = [NodeList.index(node) for node in SeedNodes]

    print(f"\n🚀 数据集: {data_name} | 节点数: {len(NodeList)} | 预算: {num_v}")

    # 注意：这里会调用 get_avg_HigerOrder_dis，请确保该函数内加载 .npy 的路径已改为 './Hyper_Distance/'
    CandidateNodes_pos = Competition_composite_indicator_new(
        SeedNodes_pos, G_matrix, data_name,
        int(CandidateNodes_Ratio * len(NodeList)), Indicator_Order, Indicator_Alpha
    ).tolist()

    # 4. 初始化 Agent
    agent = Compete_DQN(
        epsilon=epsilon_start, alpha=alpha, gamma=gamma, state_dim=state_dim,
        action_dim=len(CandidateNodes_pos),
        node_hyperedge_avgsize=get_node_hyperedge_avgsize(G_matrix),
        avg_HigerOrder_dis_seednodes=get_avg_HigerOrder_dis(SeedNodes_pos, G_matrix, data_name, Indicator_Order, Indicator_Alpha),
        candidate_nodes_pos=CandidateNodes_pos, device=device
    )
    replay_buffer = ReplayBuffer(buffer_capacity)

    # 5. 共享内存
    shm = SharedMemory(create=True, size=G_matrix.nbytes)
    try:
        shm_np = np.ndarray(G_matrix.shape, dtype=G_matrix.dtype, buffer=shm.buf)
        np.copyto(shm_np, G_matrix)

        total_steps = 0
        for episode in trange(num_episodes, desc=f"训练中"):
            agent.epsilon = max(epsilon_end, epsilon_start - episode * epsilon_decay)
            CandidateNodesPool = CandidateNodes_pos.copy()
            state_pos_list = []
            last_CompeteInf = 0
            done = False

            while not done:
                action_node, action_index = agent.take_action(state_pos_list, CandidateNodesPool)
                CandidateNodesPool.remove(action_node)

                # 仿真计算奖励
                CompeteInf = Simulations(
                    simulation_num, HG_Diffusion_SICP_pos, shm.name,
                    G_matrix.dtype, G_matrix.shape, NodeList, SeedNodes,
                    state_pos_list + [action_node], beta, T
                )

                reward = CompeteInf - last_CompeteInf
                last_CompeteInf = CompeteInf
                next_state_pos_list = state_pos_list + [action_node]
                done = (len(next_state_pos_list) == num_v)

                replay_buffer.add(agent._encode_state(state_pos_list), action_index, reward, agent._encode_state(next_state_pos_list), done)

                if len(replay_buffer) >= batch_size:
                    agent.update(list(zip(*replay_buffer.sample(batch_size))))

                if total_steps % target_update_freq == 0:
                    agent.sync_target_net()
                total_steps += 1
                state_pos_list = next_state_pos_list

        # 6. 推理与保存
        best_pos = []
        curr_state = []
        inf_pool = CandidateNodes_pos.copy()
        for _ in range(num_v):
            act = agent.best_action(curr_state, inf_pool)
            inf_pool.remove(act)
            curr_state.append(act)
            best_pos.append(act)

        res_nodes = [NodeList[idx] for idx in best_pos]
        save_path = os.path.join(CURRENT_PATH, f'selection_RL_{data_name}.npy')
        np.save(save_path, res_nodes)
        print(f"✅ 结果已存至: {save_path}")

    finally:
        shm.close()
        shm.unlink()

if __name__ == "__main__":
    seed = 42
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    
    # 7. 根据你的图片修改数据集名称 (v3000)
    datasets = [
        "synthetic_HCL_v3000_e3000_exp2.0_maxdeg10",
        "synthetic_HCL_v3000_e3000_exp3.0_maxdeg10"
    ]

    for data in datasets:
        train_and_save_CSIQ(data, beta=0.03, T=30)


🚀 数据集: synthetic_HCL_v3000_e3000_exp2.0_maxdeg10 | 节点数: 3000 | 预算: 30


KeyboardInterrupt: 

In [ ]:
import numpy as np
import os
import json
import math
import time

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import random
import matplotlib.pyplot as plt
from multiprocessing.shared_memory import SharedMemory
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import trange

# ==========================================
# ⚠️ 注意：这里删除了所有的 from XXX import YYY。
# 因为您在前面的 Cell 已经运行并定义了这些函数，
# Jupyter 的内存里已经有它们了，直接调用即可！
# ==========================================

# ==========================================
# 1. 实验全局配置
# ==========================================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['mathtext.fontset'] = 'stix'

# 根据截图，您的数据集就在当前目录下
BASE_PATH = "./" 
RESULT_DIR = "./HCL_Final_Comparison"
os.makedirs(RESULT_DIR, exist_ok=True)

HCL_DATASETS = [

    "synthetic_HCL_v3000_e3000_exp2.0_maxdeg10"
]

BETA2, BETA1, T_MAX, SIM_TIMES, SEED = 0.03, 0.05, 30, 10, 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

global_results_collector = {}
PLOT_ORDER = ['CSI-Greedy', 'HDegree', 'HADP', 'CI', 'S2V-DQN', 'CSIQ']

# S2V-DQN keeps the same training protocol as the current comparison scripts.
S2V_TRAIN_EPISODES = 100
S2V_TRAIN_SIM_TIMES = 35
S2V_ALPHA = 0.01
S2V_GAMMA = 0.8
S2V_EPSILON_START = 0.5
S2V_EPSILON_END = 0.01
S2V_BATCH_SIZE = 64
S2V_BUFFER_CAPACITY = 30000
S2V_TARGET_UPDATE_FREQ = 100
S2V_EMBEDDING_DIM = 64

# ==========================================
# 工具函数
# ==========================================
def hypergraph_to_graph(hypergraph_matrix):
    num_nodes = hypergraph_matrix.shape[0]
    OR_matrix = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    num_hyperedges = hypergraph_matrix.shape[1]
    for e in range(num_hyperedges):
        nodes_in_edge = np.where(hypergraph_matrix[:, e] > 0)[0]
        for i in nodes_in_edge:
            for j in nodes_in_edge:
                if i != j: OR_matrix[i, j] = 1.0
    return OR_matrix

# 多线程并发设计的评估函数
def evaluate_single_algo(algo_name, shm_name, dtype, shape, NodeList, seed_nodes, target_seeds_list):
    results = []
    for i in range(1, len(target_seeds_list) + 1):
        subset = target_seeds_list[:i]
        inf = Simulations(SIM_TIMES, HG_Diffusion_SICP, shm_name, dtype, shape,
                          NodeList, seed_nodes, subset, BETA2, T_MAX)
        results.append(inf)
    return algo_name, results

def train_s2v_dqn_for_eval(G_matrix, NodeList, seed_nodes, seed_pos, budget, beta, T, device, data_name):
    cache_path = os.path.join(RESULT_DIR, f"S2V_DQN_trained_nodes_({data_name})_(T_{T})_(beta_{beta}).npy")
    log_path = os.path.join(RESULT_DIR, f"S2V_DQN_train_log_({data_name})_(T_{T})_(beta_{beta}).json")
    if os.path.exists(cache_path):
        return np.load(cache_path, allow_pickle=True).tolist()[:budget]

    graph_adj = hypergraph_to_graph(G_matrix)
    candidate_pos = [node for node in range(len(NodeList)) if node not in seed_pos]
    agent = Compete_S2V_DQN(
        epsilon=S2V_EPSILON_START,
        alpha=S2V_ALPHA,
        gamma=S2V_GAMMA,
        adj_matrix=graph_adj,
        edge_weights=graph_adj,
        nodes_number=len(NodeList),
        candidate_pos=candidate_pos,
        device=device,
        embedding_dim=S2V_EMBEDDING_DIM,
    )
    replay_buffer = S2VReplayBuffer(S2V_BUFFER_CAPACITY)
    epsilon_decay = (S2V_EPSILON_START - S2V_EPSILON_END) / S2V_TRAIN_EPISODES

    returns, losses = [], []
    total_steps = 0
    shm = SharedMemory(create=True, size=G_matrix.nbytes)
    try:
        shm_np = np.ndarray(G_matrix.shape, dtype=G_matrix.dtype, buffer=shm.buf)
        np.copyto(shm_np, G_matrix)

        for episode in trange(S2V_TRAIN_EPISODES, desc=f"S2V-DQN Training on {data_name}"):
            agent.epsilon = max(S2V_EPSILON_END, S2V_EPSILON_START - episode * epsilon_decay)
            candidate_pool = set(candidate_pos.copy())
            state = []
            last_influence = 0.0
            episode_return = 0.0
            episode_losses = []
            done = False

            while not done and candidate_pool:
                action = agent.take_action(state, list(candidate_pool))
                candidate_pool.discard(action)

                influence = Simulations(
                    S2V_TRAIN_SIM_TIMES,
                    HG_Diffusion_SICP_pos,
                    shm.name,
                    G_matrix.dtype,
                    G_matrix.shape,
                    NodeList,
                    seed_nodes,
                    state + [action],
                    beta,
                    T,
                )
                reward = influence - last_influence
                last_influence = influence
                episode_return += reward

                next_state = state + [action]
                done = len(next_state) >= budget
                replay_buffer.add(state, action, reward, next_state, done)

                if len(replay_buffer) >= S2V_BATCH_SIZE:
                    batch = replay_buffer.sample(S2V_BATCH_SIZE)
                    episode_losses.append(float(agent.update(batch)))

                if total_steps % S2V_TARGET_UPDATE_FREQ == 0:
                    agent.sync_target_net()
                total_steps += 1
                state = next_state

            returns.append(float(episode_return))
            losses.append(float(np.mean(episode_losses)) if episode_losses else 0.0)
    finally:
        shm.close()
        shm.unlink()

    selected_pos = []
    temp_candidates = candidate_pos.copy()
    while len(selected_pos) < budget and temp_candidates:
        best_node = agent.best_action(selected_pos, temp_candidates)
        selected_pos.append(best_node)
        temp_candidates.remove(best_node)

    selected_nodes = [NodeList[i] for i in selected_pos]
    np.save(cache_path, np.array(selected_nodes, dtype=object))
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump({
            'data_name': data_name,
            'beta': beta,
            'T': T,
            'train_episodes': S2V_TRAIN_EPISODES,
            'train_sim_times': S2V_TRAIN_SIM_TIMES,
            'returns': returns,
            'losses': losses,
            'nodes': selected_nodes,
        }, f, indent=4)
    return selected_nodes

def get_style(name):
    styles = {
        'CSI-Greedy': {'color': '#ff7f0e', 'marker': 's', 'z': 5},
        'HDegree': {'color': '#EC6E66', 'marker': 'd', 'z': 2},
        'HADP': {'color': '#FAC7B3', 'marker': '^', 'z': 2},
        'CI': {'color': '#91CCC0', 'marker': '*', 'z': 2},
        'S2V-DQN': {'color': '#2ca02c', 'marker': 'v', 'z': 5},
        'CSIQ': {'color': '#1f77b4', 'marker': '8', 'z': 10}
    }
    return styles.get(name, {'color': 'gray', 'marker': 'o', 'z': 1})

# ==========================================
# 主流程
# ==========================================
def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    for idx, data_name in enumerate(HCL_DATASETS):
        print(f"\n{'='*50}\n🚀 开始处理数据集 ({idx + 1}/{len(HCL_DATASETS)}): {data_name}")
        
        # -----------------------------------------------------
        # 【核心优化】前置防御性检查：开始漫长计算前，先确认所有文件就绪！
        # -----------------------------------------------------
        m_p = os.path.join(BASE_PATH, f"{data_name}.npy")
        n_p = os.path.join(BASE_PATH, f"{data_name}_nodelist.npy")
        csiq_file_path = os.path.join(BASE_PATH, "New_List", f"indicator_selection_RL_11_({data_name})_(T_{T_MAX})_(beta_{BETA2})_(ratio_0.2).npy")
        
        if not os.path.exists(m_p) or not os.path.exists(n_p): 
            print(f"  ❌ 找不到底层数据集文件 (.npy)，跳过该数据集...")
            continue
            
        if not os.path.exists(csiq_file_path):
            print(f"  ❌ 未找到 CSIQ 预计算结果: {csiq_file_path}")
            print(f"  ⚠️ 为防止白白浪费数小时计算基线算法，已强制跳过该数据集！请先完成该数据的 RL 训练。")
            continue
            
        print(f"  ✅ 所有必要文件(含 CSIQ 结果)检查完毕，开始执行评估计算...")

        # 开始正式处理
        G_matrix = np.load(m_p, allow_pickle=True)
        NodeList = np.load(n_p, allow_pickle=True).tolist()
        num_v = int(0.01 * len(NodeList))

        Vs_names = H_Degree_SG(G_matrix, NodeList, num_v)
        Vs_pos = [NodeList.index(node) for node in Vs_names]

        algo_seeds = {}
        
        # 既然前面已经检查过了，这里直接加载 CSIQ 结果
        algo_seeds['CSIQ'] = np.load(csiq_file_path, allow_pickle=True).tolist()

        print("  [1/4] 正在计算 CSI 指标及运行 CSI-Greedy (耗时较长)...")
        cand_pos_list = Competition_composite_indicator_new(Vs_pos, G_matrix, data_name,
                                                            int(0.2 * len(NodeList)), 10, 0.5).tolist()
        algo_seeds['CSI-Greedy'] = Indicator_selection_Greedy(G_matrix, NodeList, Vs_names,
                                                              cand_pos_list, num_v, 100, BETA2, T_MAX)

        print("  [2/4] 正在运行启发式基线 (HDegree, HADP, CI)...")
        algo_seeds['HDegree'] = H_Degree(G_matrix, NodeList, Vs_names, num_v)
        algo_seeds['HADP'] = HADP(G_matrix, NodeList, Vs_names, num_v)
        algo_seeds['CI'] = CI(G_matrix, NodeList, Vs_names, num_v)

        print("  [3/4] ??????? S2V-DQN...")
        algo_seeds['S2V-DQN'] = train_s2v_dqn_for_eval(
            G_matrix,
            NodeList,
            Vs_names,
            Vs_pos,
            num_v,
            BETA2,
            T_MAX,
            device,
            data_name,
        )

        print("  [4/4] 所有算法种子挑选完成，准备进入综合影响力评估...")

        # -----------------------------------------------------
        # 并发极速评估引擎 (多线程 + 唯一共享内存)
        # -----------------------------------------------------
        print(f"\n  ⚡ 开启多线程并发评估 (并发数: {len(algo_seeds)})...")
        eval_start_time = time.time()
        current_data_results = {}
        
        shm = SharedMemory(create=True, size=G_matrix.nbytes)
        try:
            shm_np = np.ndarray(G_matrix.shape, dtype=G_matrix.dtype, buffer=shm.buf)
            np.copyto(shm_np, G_matrix)

            with ThreadPoolExecutor(max_workers=len(PLOT_ORDER)) as executor:
                futures = []
                for name in PLOT_ORDER:
                    if name in algo_seeds:
                        futures.append(
                            executor.submit(evaluate_single_algo, name, shm.name, 
                                            G_matrix.dtype, G_matrix.shape, 
                                            NodeList, Vs_names, algo_seeds[name])
                        )
                
                for future in as_completed(futures):
                    try:
                        algo_name, res = future.result()
                        current_data_results[algo_name] = res
                        print(f"    ✅ {algo_name} 评估完成!")
                    except Exception as exc:
                        print(f"    ❌ 评估发生异常: {exc}")
                        
        finally:
            shm.close()
            shm.unlink()
            
        print(f"  ⏱️ 本数据集评估总耗时: {time.time() - eval_start_time:.2f} 秒")

        global_results_collector[data_name] = current_data_results

        # -----------------------------------------------------
        # 将该数据集的画图数据持久化保存为 JSON
        # -----------------------------------------------------
        json_friendly_results = {k: [float(val) for val in v] for k, v in current_data_results.items()}
        json_path = os.path.join(RESULT_DIR, f"{data_name}_plot_data.json")
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(json_friendly_results, f, indent=4)
        print(f"  💾 画图数据已备份至: {json_path}")

    # -----------------------------------------------------
    # 汇总大图绘制 
    # -----------------------------------------------------
    if not global_results_collector:
        print("\n⚠️ 警告: 没有收集到任何评估数据，可能所有数据集都因为缺少文件被跳过了。画图终止。")
        return

    print("\n📈 正在绘制精美对比图表...")
    
    num_plots = len(global_results_collector)
    cols = 3
    rows = math.ceil(num_plots / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

    for idx, (data_name, data_results) in enumerate(global_results_collector.items()):
        ax = axes[idx]
        
        parts = data_name.split('_')
        title_suffix = f"{parts[2]}_{parts[4]}" if len(parts) > 4 else data_name
        ax.set_title(f"HCL ({title_suffix})", fontsize=14, fontweight='bold')
        
        fig_single, ax_single = plt.subplots(figsize=(6, 5))
        ax_single.set_title(f"HCL ({title_suffix})", fontsize=14, fontweight='bold')

        for algo_name in PLOT_ORDER:
            if algo_name in data_results:
                y_values = data_results[algo_name]
                x_values = range(1, len(y_values) + 1)
                style = get_style(algo_name)
                
                ax.plot(x_values, y_values, label=algo_name, 
                        color=style['color'], marker=style['marker'], 
                        linewidth=2.5 if algo_name == 'CSIQ' else 1.5,
                        markersize=8 if algo_name == 'CSIQ' else 6,
                        zorder=style['z'])
                
                ax_single.plot(x_values, y_values, label=algo_name, 
                               color=style['color'], marker=style['marker'], 
                               linewidth=2.5 if algo_name == 'CSIQ' else 1.5,
                               markersize=8 if algo_name == 'CSIQ' else 6,
                               zorder=style['z'])

        ax.set_xlabel('Budget (b)', fontsize=12)
        ax.set_ylabel('Target Expected Influence', fontsize=12)
        ax.grid(True, linestyle='--', alpha=0.6)
        if idx == 0: ax.legend(loc='best', fontsize=11, framealpha=0.9)
        
        ax_single.set_xlabel('Budget (b)', fontsize=12)
        ax_single.set_ylabel('Target Expected Influence', fontsize=12)
        ax_single.grid(True, linestyle='--', alpha=0.6)
        ax_single.legend(loc='best', fontsize=11, framealpha=0.9)
        fig_single.tight_layout()
        single_save_path = os.path.join(RESULT_DIR, f"{data_name}_Individual_Plot.pdf")
        fig_single.savefig(single_save_path, format='pdf', bbox_inches='tight')
        plt.close(fig_single) 

    for i in range(num_plots, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    combined_save_path = os.path.join(RESULT_DIR, 'HCL_All_Algorithms_Comparison.pdf')
    plt.savefig(combined_save_path, format='pdf', bbox_inches='tight')
    print(f"✅ 绘图完成！大组合图表已保存至: {combined_save_path}")
    
    plt.show()

# ==========================================
# 执行入口
# ==========================================
if __name__ == "__main__":
    main()


🚀 开始处理数据集 (1/1): synthetic_HCL_v3000_e3000_exp2.0_maxdeg10
  ✅ 所有必要文件(含 CSIQ 结果)检查完毕，开始执行评估计算...
  [1/4] 正在计算 CSI 指标及运行 CSI-Greedy (耗时较长)...


indicator_selection_greedy: 100%|██████████| 30/30 [43:30:03<00:00, 5220.13s/it]   


  [2/4] 正在运行启发式基线 (HDegree, HADP, CI)...


100%|██████████| 3000/3000 [00:00<00:00, 5008.94it/s]


  [3/4] 正在运行 S2V-DQN...
  [4/4] 所有算法种子挑选完成，准备进入综合影响力评估...

  ⚡ 开启多线程并发评估 (并发数: 6)...
    ✅ CI 评估完成!
    ✅ CSI-Greedy 评估完成!
    ✅ HDegree 评估完成!
    ✅ HADP 评估完成!
    ✅ S2V-DQN 评估完成!
    ✅ CSIQ 评估完成!
  ⏱️ 本数据集评估总耗时: 158.15 秒
  💾 画图数据已备份至: ./HCL_Final_Comparison/synthetic_HCL_v3000_e3000_exp2.0_maxdeg10_plot_data.json

📈 正在绘制精美对比图表...


findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: 

findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: Generic family 'serif' not found because none of the following families were found: Times New Roman
findfont: 